# Paper 3 Low-Budget KCD Runner

This notebook downloads or normalizes smaller conversational-memory benchmarks and runs the new low-budget Paper 3 policy family:

- `uniform`
- `semantic`
- `geometry`
- `geometry_keep_compress_drop`
- `support_aware_geometry_keep_compress_drop`
- `semantic_keep_compress_drop`
- `semantic_filtered_geometry_keep_compress_drop`


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/SteveMama/rt-geometry-memory.git"
REPO_DIR = "/content/rt-geometry-memory"

def run_streaming(cmd, cwd=None):
    print("RUN", " ".join(cmd))
    process = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)

%cd /content
if os.path.exists(REPO_DIR):
    run_streaming(["git", "-C", REPO_DIR, "pull"])
else:
    run_streaming(["git", "clone", REPO_URL, REPO_DIR])
%cd /content/rt-geometry-memory
run_streaming(["bash", "scripts/colab_setup.sh"], cwd=REPO_DIR)


In [ ]:
BATCH_PREFIX = "paper3_low_budget_v1"
MODEL_KEYS = "qwen25_15b"
BUDGETS = "0.20,0.35,0.50"
LIMIT_CONVERSATIONS = 24
TARGET_TURN_STRIDE = 4
MAX_TARGET_TURNS = 16
BENCHMARK_NAMES = ["msc_valid", "locomo10"]
MANUAL_SOURCE_PATH = "/content/manual_benchmark_source.jsonl"


In [ ]:
BENCHMARK_CONFIG = {
    "msc_train": {"format": "msc", "source": "/content/msc_train.jsonl", "auto_download": True},
    "msc_valid": {"format": "msc", "source": "/content/msc_valid.jsonl", "auto_download": True},
    "msc_test": {"format": "msc", "source": "/content/msc_test.jsonl", "auto_download": True},
    "locomo10": {"format": "locomo", "source": "/content/locomo10.json", "auto_download": True},
    "longmemeval_s_cleaned": {"format": "longmemeval", "source": "/content/longmemeval_s_cleaned.json", "auto_download": True},
    "gapchat_manual": {"format": "normalized", "source": MANUAL_SOURCE_PATH, "auto_download": False},
    "realtalk_manual": {"format": "normalized", "source": MANUAL_SOURCE_PATH, "auto_download": False},
    "evolmem_manual": {"format": "normalized", "source": MANUAL_SOURCE_PATH, "auto_download": False},
    "normalized_manual": {"format": "normalized", "source": MANUAL_SOURCE_PATH, "auto_download": False},
}

def normalized_output_path(name: str) -> str:
    return f"{REPO_DIR}/benchmarks/{name}_normalized.jsonl"


In [ ]:
for benchmark_name in BENCHMARK_NAMES:
    cfg = BENCHMARK_CONFIG[benchmark_name]
    if cfg["auto_download"]:
        run_streaming(
            [
                "python",
                "scripts/download_public_benchmark.py",
                "--benchmark",
                benchmark_name,
                "--output",
                cfg["source"],
            ],
            cwd=REPO_DIR,
        )
    run_streaming(
        [
            "python",
            "scripts/prepare_public_benchmark_jsonl.py",
            "--format",
            cfg["format"],
            "--input",
            cfg["source"],
            "--output",
            normalized_output_path(benchmark_name),
            "--family",
            benchmark_name,
        ],
        cwd=REPO_DIR,
    )


In [ ]:
for benchmark_name in BENCHMARK_NAMES:
    run_streaming(
        [
            "bash",
            "scripts/run_paper3_low_budget_kcd_probe.sh",
            f"{BATCH_PREFIX}_{benchmark_name}",
            normalized_output_path(benchmark_name),
            MODEL_KEYS,
            BUDGETS,
            str(LIMIT_CONVERSATIONS),
            str(TARGET_TURN_STRIDE),
            str(MAX_TARGET_TURNS),
        ],
        cwd=REPO_DIR,
    )


In [ ]:
for benchmark_name in BENCHMARK_NAMES:
    report_dir = Path(REPO_DIR) / "results" / "paper3" / "studies" / f"{BATCH_PREFIX}_{benchmark_name}"
    print("\n===", benchmark_name, "===")
    print((report_dir / "study_report.md").read_text(encoding="utf-8")[:4000])
    pairwise_path = report_dir / "pairwise_report.md"
    if pairwise_path.exists():
        print(pairwise_path.read_text(encoding="utf-8")[:3000])
